In [3]:
# Load CSVs (change the date folder in one place only)
import pandas as pd
from pathlib import Path

base = Path('../CSV/25.10.2026')
paths = {
    'game_week_scores': base / 'game_week_scores_rows.csv',
    'game_weeks': base / 'game_weeks_rows.csv',
    'profiles': base / 'profiles_rows.csv',
    'seasons': base / 'seasons_rows.csv',
}

def load_csv(p):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f'File not found: {p}')
    df = pd.read_csv(p, dtype=str)
    return df.apply(lambda col: col.str.strip())

gw_scores = load_csv(paths['game_week_scores'])
gws = load_csv(paths['game_weeks'])
profiles = load_csv(paths['profiles'])
seasons = load_csv(paths['seasons'])

for c in ('points', 'correct_scores'):
    gw_scores[c] = pd.to_numeric(gw_scores[c], errors='coerce').fillna(0).astype(int)

# game_week_scores -> game_weeks (for season_id) -> profiles (for username)
merged = (
    gw_scores
    .merge(gws[['id', 'season_id']].rename(columns={'id': 'game_week_id'}), on='game_week_id', how='left')
    .merge(profiles[['id', 'username']].rename(columns={'id': 'player_id'}), on='player_id', how='left')
)
merged['username'] = merged['username'].fillna(merged['player_id'])

print('Rows:', len(merged), '| Seasons:', ', '.join(seasons['name']))

Rows: 1699 | Seasons: 2025/26, 2026/27


In [4]:
# Season table with a season drop-down
import ipywidgets as widgets
from IPython.display import display

# Newest season first (default selection)
seasons_sorted = seasons.sort_values('start_date', ascending=False)
options = [(row['name'], row['id']) for _, row in seasons_sorted.iterrows()]

def season_table(season_id):
    df = merged[merged['season_id'] == season_id]
    table = (
        df.groupby('player_id')
        .agg(Player=('username', 'first'), game_weeks_played=('game_week_id', 'nunique'), correct_scores=('correct_scores', 'sum'), points=('points', 'sum'))
        .sort_values(['points', 'correct_scores'], ascending=False)
        .reset_index(drop=True)
    )
    table.index = table.index + 1
    table.index.name = 'Pos'
    return table.rename(columns={'correct_scores': 'Correct Scores', 'points': 'Points'})

dropdown = widgets.Dropdown(options=options, description='Season:')
out = widgets.Output()

def refresh(*_):
    with out:
        out.clear_output(wait=True)
        display(season_table(dropdown.value))

dropdown.observe(refresh, names='value')
display(dropdown, out)
refresh()

Dropdown(description='Season:', options=(('2026/27', 'f010783f-7fb5-40d8-9e00-6d8a45fa448e'), ('2025/26', '274…

Output()